# 01 - Data and the split

Two jobs: get HAM10000 into `data/`, and create the train/val/test split that all
twelve experiments share.

The split is lesion-level, not image-level. HAM10000 contains several photographs of
the same lesion, so splitting on images would put near-duplicates on both sides of the
train/test boundary and quietly inflate every number in the study. Splitting on
`lesion_id` keeps all photographs of a lesion in one partition.

Seed is fixed at 42 and the result is hashed. Every training notebook prints that hash,
so if one run ever saw different data it shows up immediately.

In [1]:
import json, sys, shutil, zipfile
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "ham10000").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA = ROOT / "data"
RUNS = ROOT / "runs"
print("repo:", ROOT)

repo: /workspace/ham10000-cnn-comparison


## Source

Downloaded from Harvard Dataverse, which is the citable source:

> Tschandl, P. *The HAM10000 dataset, a large collection of multi-source dermatoscopic
> images of common pigmented skin lesions*. Harvard Dataverse, V4 (2018).
> https://doi.org/10.7910/DVN/DBW86T

Two things about that distribution are worth knowing, because both will break a naive
loader:

1. The images arrive as two zips with the JPEGs flat at the root, not in folders.
2. The metadata file is called `HAM10000_metadata`, with no `.csv` extension, even
   though its contents are ordinary CSV.

The cell below handles both. Point `DATAVERSE` at wherever the download landed.

In [2]:
# The dataset ships inside this repo under dataverse_files/. The other
# candidates are fallbacks for running outside the packaged layout.
CANDIDATES = [
    ROOT / "dataverse_files",
    Path.home() / "Downloads" / "dataverse_files",
    Path.home() / "Desktop" / "Desktop_stuff" / "Thesis_Theer" / "dataverse_files",
]

DATAVERSE = next((c for c in CANDIDATES if c.exists()), CANDIDATES[0])

print("source:", DATAVERSE, "->", "found" if DATAVERSE.exists() else "NOT FOUND")
if DATAVERSE.exists():
    for f in sorted(DATAVERSE.iterdir()):
        print(f"  {f.name:<55} {f.stat().st_size/1e6:>9.1f} MB")

source: /workspace/ham10000-cnn-comparison/dataverse_files -> found
  HAM10000_images_part_1.zip                                 1366.5 MB
  HAM10000_images_part_2.zip                                 1403.6 MB
  HAM10000_metadata                                             0.7 MB


### Unpack

Roughly 2.7 GB of JPEGs, so give this a few minutes. It is a no-op if the files are
already in place, which makes the notebook safe to re-run.

In [10]:
DATA.mkdir(parents=True, exist_ok=True)

# 1. metadata: copy across and give it the .csv extension the loader expects
meta_path = DATA / "HAM10000_metadata.csv"
if not meta_path.exists():
    src = DATAVERSE / "HAM10000_metadata"
    if not src.exists():
        src = DATAVERSE / "HAM10000_metadata.csv"
    shutil.copy(src, meta_path)
    print("copied metadata ->", meta_path.name)
else:
    print("metadata already in place")

# 2. images: each zip is flat, so extract into its own named folder
for part in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    target = DATA / part
    existing = len(list(target.glob("*.jpg"))) if target.exists() else 0
    if existing:
        print(f"{part}: {existing} images already extracted")
        continue
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATAVERSE / f"{part}.zip") as z:
        z.extractall(target)
    print(f"{part}: extracted {len(list(target.glob('*.jpg')))} images")

metadata already in place
HAM10000_images_part_1: 5000 images already extracted
HAM10000_images_part_2: 5015 images already extracted


### Check what landed

The two folders together must hold 10,015 JPEGs, and the metadata must have one row
per image. If either assertion trips, stop and fix the download rather than training on
a partial dataset.

In [11]:
import pandas as pd

assert meta_path.exists(), f"{meta_path} missing"
meta = pd.read_csv(meta_path)

counts = {p: len(list((DATA / p).glob("*.jpg")))
          for p in ["HAM10000_images_part_1", "HAM10000_images_part_2"]}
total_images = sum(counts.values())

print(counts)
print(f"images on disk : {total_images}")
print(f"metadata rows  : {len(meta)}")
print(f"unique lesions : {meta.lesion_id.nunique()}")
print(f"columns        : {list(meta.columns)}")

assert total_images == 10015, f"expected 10015 images, found {total_images}"
assert len(meta) == 10015, f"expected 10015 metadata rows, found {len(meta)}"

{'HAM10000_images_part_1': 5000, 'HAM10000_images_part_2': 5015}
images on disk : 10015
metadata rows  : 10015
unique lesions : 7470
columns        : ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset']


Every `image_id` in the metadata must actually resolve to a file. A missing
image would otherwise surface as a crash partway through the first epoch.

In [12]:
from ham10000.dataset import resolve_image_path

missing = []
for iid in meta.image_id:
    try:
        resolve_image_path(DATA, iid)
    except FileNotFoundError:
        missing.append(iid)

print(f"unresolvable image_ids: {len(missing)}")
assert not missing, missing[:10]
print("every metadata row maps to a file on disk")

unresolvable image_ids: 0
every metadata row maps to a file on disk


## Stage the images into RAM

`/workspace` is a network volume, and reading 8,012 JPEGs across it one file at a time
was the limiting factor: 45 images/second there against 241 from RAM. The pod has 29 GB
of `/dev/shm` and the dataset is 2.6 GB, so the whole thing fits.

Unzipping from the archives beats copying the extracted folder, because that reads two
large files sequentially rather than 10,015 small ones over FUSE.

This is a pure speed change. Same bytes, same order, same seeds, so it has no effect on
any result. `/dev/shm` is cleared when the pod restarts, so re-run this cell if that
happens; the training notebooks fall back to the on-disk copy if it is gone.

In [ ]:
import shutil, subprocess, time

FAST = Path("/dev/shm/ham_data")
need = not (FAST / "HAM10000_metadata.csv").exists() or \
       len(list((FAST / "HAM10000_images_part_2").glob("*.jpg"))) != 5015

if need:
    t = time.time()
    for part in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        (FAST / part).mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o",
                        str(DATAVERSE / f"{part}.zip"), "-d", str(FAST / part)], check=True)
    shutil.copy(DATA / "HAM10000_metadata.csv", FAST / "HAM10000_metadata.csv")
    print(f"staged in {time.time() - t:.0f}s")
else:
    print("already staged")

for part in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    print(f"  {part}: {len(list((FAST / part).glob('*.jpg')))} images")
print(f"\nfree in /dev/shm: {shutil.disk_usage('/dev/shm').free / 1e9:.1f} GB")

## Build the split

80/10/10 over lesions, stratified by diagnosis. Created once here, then loaded by all
twelve training notebooks.

In [13]:
from ham10000.split import load_or_create_split, split_hash

split_path = RUNS / "splits" / "seed42_lesion_stratified.json"
df, lesion_splits = load_or_create_split(meta_path, split_path)

print("split file :", split_path)
print("hash       :", split_hash(lesion_splits))
print()
print(df.split.value_counts().reindex(["train", "val", "test"]).rename("images"))
print()
print({k: len(v) for k, v in lesion_splits.items()}, "lesions")

split file : /workspace/ham10000-cnn-comparison/runs/splits/seed42_lesion_stratified.json
hash       : f35ac1de18678182

split
train    8012
val       979
test     1024
Name: images, dtype: int64

{'train': 5976, 'val': 747, 'test': 747} lesions


### Two things that have to hold

No lesion may appear in more than one partition, and the class proportions should
survive the split. If the first check fails, every downstream result is contaminated
and nothing else in the repository is trustworthy.

In [14]:
sets = {k: set(v) for k, v in lesion_splits.items()}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = sets[a] & sets[b]
    print(f"{a:>5} n {b:<5} overlap: {len(overlap)}")
    assert not overlap, f"lesion leak between {a} and {b}"

# The same check at image level, which is what actually matters for leakage.
img_sets = {s: set(df[df.split == s].image_id) for s in ["train", "val", "test"]}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not (img_sets[a] & img_sets[b]), f"image leak between {a} and {b}"
print("\nno lesion or image appears in two partitions")

train n val   overlap: 0
train n test  overlap: 0
  val n test  overlap: 0

no lesion or image appears in two partitions


In [15]:
proportions = (df.groupby("split").dx.value_counts(normalize=True)
                 .unstack().mul(100).round(1)
                 .reindex(["train", "val", "test"]))
print("class share per partition (%)")
print(proportions.T)
print()
print("absolute counts")
print(df.groupby("split").dx.value_counts().unstack()
        .reindex(["train", "val", "test"]).T.fillna(0).astype(int))

class share per partition (%)
split  train   val  test
dx                      
akiec    3.3   3.4   3.2
bcc      5.0   5.9   5.2
bkl     11.2  10.0  10.3
df       1.1   1.1   1.3
mel     11.1  10.8  11.2
nv      66.8  67.5  67.4
vasc     1.4   1.2   1.5

absolute counts
split  train  val  test
dx                     
akiec    261   33    33
bcc      403   58    53
bkl      896   98   105
df        91   11    13
mel      892  106   115
nv      5354  661   690
vasc     115   12    15


### Record what the test set contains

These support counts are quoted throughout the results chapter, so they are printed
here once, from the split itself, rather than being remembered.

In [16]:
test = df[df.split == "test"]
print(f"test set: {len(test)} images from {test.lesion_id.nunique()} lesions\n")
for dx, n in test.dx.value_counts().items():
    print(f"  {dx:<6} {n:>4}")

test set: 1024 images from 747 lesions

  nv      690
  mel     115
  bkl     105
  bcc      53
  akiec    33
  vasc     15
  df       13
